# DemandIQ — Stage 3: Full training, exploration, and export (Colab)

This notebook runs the **same model the backend serves**, end to end, on the **real Recruit Restaurant Visitor Forecasting dataset** from Kaggle.

It is the executable companion to the pipeline docs in `model/PIPELINE.md` and the Colab runbook in `model/COLAB.md`. The code does not invent a workflow; it drives the existing `diq_model` package — the same package the API loads a bundle from — so the artifacts this notebook produces are the artifacts the backend is ready to consume.

**What this notebook does**

1. **Config** — detect Colab vs local, mount Drive, point at data / checkpoints / figures.
2. **Install** — install `diq_model` with the training extras into the running kernel.
3. **Data** — fetch the 4 real Recruit CSVs from Kaggle (idempotent), then audit them.
4. **Exploration** — a long, narrated data-exploration pass over the raw data and the leakage-safe feature frame.
5. **Train** — the full pipeline: `ingest → baseline → features → fallbacks → lightgbm → catboost → conformal → policy → shap → sensitivity → backtest`, checkpointed and resumable.
6. **Report** — mirror the results chapter: per-model metrics table, baseline comparison, intervals, policy trade-off, SHAP top features, provenance.
7. **Export** — write the exact `serving_bundle.joblib` the API loads, and gather the downloadable artifacts.

**Proposal alignment**

`ingest`/`audit` = Sections 3.4, 3.7; split = Section 3.11 chronological; metrics = Section 3.15; intervals = Section 3.16; policy = Sections 3.17–3.20; explanations = Section 3.24; sensitivity = Section 3.25; backtest = Section 3.25 rolling-origin.

**Runtime reality**

The default run is 9 stages plus the two opt-in stages (`sensitivity`, `backtest`). On Colab this is typically a few minutes on CPU; the heavy cells are features, fallbacks, conformal, SHAP, and the backtest refits. Every stage writes an atomic checkpoint, so a runtime recycle is survived by re-running **Run all** — completed stages are skipped and reloaded from Drive.


## 0. Config — the only cell you normally need to edit

Same code locally and on Colab; only the paths differ. On Colab, artifacts and data live in Drive so a recycled runtime does not lose them. On local it uses `model/data/` and writes to `model/artifacts/runs/nb03`.


In [ ]:
# ==============================================================================
# EDITABLE BOUNDARIES:
#   IS_COLAB         - auto-detected; you can force it True/False for testing.
#   REPO_ROOT        - where the repo was cloned on the VM.
#   DATA_ROOT        - the folder with the 4 Recruit CSVs.
#   CKPT_DIR         - where stages + serving_bundle.joblib are written.
#   TEST_START/SEED  - these are the exact constants train.py uses.
#   STAGES           - remove 'sensitivity'/'backtest' here to shorten a run.
#   FORCE           - False = resume from checkpoints; True = rerun all.
#   STRICT          - True = block on any audit deviation; False = report only.
# The rest of the notebook reads from these variables, never from hidden state.
# EDITABLE BOUNDARIES:
#   IS_COLAB         - auto-detected; you can force it True/False for testing.
#   REPO_ROOT        - where the repo was cloned on the VM.
#   DATA_ROOT        - the folder with the 4 Recruit CSVs.
#   CKPT_DIR         - where stages + serving_bundle.joblib are written.
#   TEST_START/SEED  - these are the exact constants train.py uses.
#   STAGES           - remove 'sensitivity'/'backtest' here to shorten a run.
#   FORCE           - False = resume from checkpoints; True = rerun all.
#   STRICT          - True = block on any audit deviation; False = report only.
# The rest of the notebook reads from these variables, never from hidden state.
# CONFIG — edit the paths / flags here, then Run all.
# ==============================================================================
from pathlib import Path
import os
import sys

# ---------------------------------------------------------------- environment
IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/demandiq")
    DATA_ROOT = Path("/content/drive/MyDrive/demandiq/data/recruit")
    CKPT_DIR = Path("/content/drive/MyDrive/demandiq/artifacts/runs/nb03")
    FIG_DIR = Path("/content/drive/MyDrive/demandiq/figures/nb03")
else:
    def _find_repo_root() -> Path:
        for candidate in [Path.cwd(), *Path.cwd().parents]:
            if (candidate / "model" / "pyproject.toml").exists():
                return candidate
        return Path.cwd()
    REPO_ROOT = _find_repo_root()
    DATA_ROOT = REPO_ROOT / "model" / "data" / "recruit"
    CKPT_DIR = REPO_ROOT / "model" / "artifacts" / "runs" / "nb03"
    FIG_DIR = REPO_ROOT / "model" / "figures" / "nb03"

FIG_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle slug (Section 3.4.1).
KAGGLE_COMPETITION = "recruit-restaurant-visitor-forecasting"
RECRUIT_CSVS = ("air_visit_data.csv", "air_store_info.csv", "date_info.csv", "air_reserve.csv")

# Held-out test window start (Section 3.11). Default matches the Recruit competition split.
TEST_START = "2017-02-01"
SEED = 42

# Full pipeline, including the two opt-in stages.
STAGES = [
    "ingest",
    "baseline",
    "features",
    "fallbacks",
    "lightgbm",
    "catboost",
    "conformal",
    "policy",
    "shap",
    "sensitivity",
    "backtest",
]

# Resumability. False skips checkpointed stages; True recomputes everything.
FORCE = False

# Audit strictness. True blocks on every deviation from the audited figures;
# False reports them without blocking (used for subset / smoke runs).
STRICT = True

# Backtest configuration (Section 3.25). Each fold refits LightGBM from scratch.
BACKTEST_FOLDS = 6
BACKTEST_TEST_DAYS = 30
BACKTEST_ADVANCE_DAYS = 14
BACKTEST_MIN_TRAIN_DAYS = 180

print("environment :", "colab" if IS_COLAB else "local")
print("repo        :", REPO_ROOT)
print("data        :", DATA_ROOT)
print("checkpoints :", CKPT_DIR)
print("figures     :", FIG_DIR)
print("stages      :", len(STAGES), "->", ", ".join(STAGES))
print("test_start  :", TEST_START, "| seed", SEED)


## 1. Install the package

`pyarrow` is a hard requirement because every stage writes parquet checkpoints. `[training]` pulls LightGBM, CatBoost, XGBoost, scikit-learn and SHAP.

Colab ships LightGBM but not CatBoost or SHAP, so the extra is required there too. The install is idempotent and runs through `sys.executable`, never a shell escape, so it works in the same kernel that runs the rest of the notebook.


In [ ]:
# ==============================================================================
# WHAT THIS CELL DOES:
#   1. Checks whether diq_model + pyarrow + core libs are already importable.
#   2. If not, pip-installs the editable model package with [training] extras.
#   3. Puts model/src on sys.path so the just-installed package is visible.
#   4. Asserts diq_model resolved from THIS repo, not some stray copy.
#   5. Drops stages whose deps (sklearn/lightgbm/catboost/shap) are unusable.
#
# WHY EDITABLE INSTALL:
#   The notebook does not bundle a wheel; it runs the code in this repo.
#   Editable install means changes to model/src/ are picked up on the next run.
#
# WHY THE ASSERT MATTERS:
#   Without it, import diq_model could silently resolve to an old copy elsewhere
#   on sys.path, and you would train a different codebase than the one you edited.
# WHAT THIS CELL DOES:
#   1. Checks whether diq_model + pyarrow + core libs are already importable.
#   2. If not, pip-installs the editable model package with [training] extras.
#   3. Puts model/src on sys.path so the just-installed package is visible.
#   4. Asserts diq_model resolved from THIS repo, not some stray copy.
#   5. Drops stages whose deps (sklearn/lightgbm/catboost/shap) are unusable.
#
# WHY EDITABLE INSTALL:
#   The notebook does not bundle a wheel; it runs the code in this repo.
#   Editable install means changes to model/src/ are picked up on the next run.
#
# WHY THE ASSERT MATTERS:
#   Without it, import diq_model could silently resolve to an old copy elsewhere
#   on sys.path, and you would train a different codebase than the one you edited.
# INSTALL — core + training extras + pyarrow.
# ==============================================================================
import importlib
import importlib.util
import subprocess
import sys

PKG_SPEC = f"{REPO_ROOT / 'model'}[training]"

CORE = {
    "pandas":   "frames",
    "numpy":    "frames",
    "pyarrow":  "parquet checkpoints",
    "joblib":   "model persistence",
    "matplotlib": "exploration figures",
}
STAGE_DEPS = {
    "fallbacks": ("sklearn",),
    "lightgbm":  ("lightgbm",),
    "catboost":  ("catboost",),
    "shap":      ("shap",),
}

def probe(name: str) -> bool:
    if importlib.util.find_spec(name) is None:
        return False
    try:
        importlib.import_module(name)
        return True
    except BaseException:
        return False

need_install = (not probe("diq_model")) or (not probe("pyarrow")) or any(
    not probe(n) for n in CORE
)

if need_install:
    print(f"installing {PKG_SPEC} + pyarrow into {sys.executable} ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", PKG_SPEC, "pyarrow"],
        check=True,
    )
    src_dir = REPO_ROOT / "model" / "src"
    if src_dir.is_dir() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))
    importlib.invalidate_caches()

import diq_model
assert Path(diq_model.__file__).resolve().is_relative_to(REPO_ROOT.resolve()), (
    f"diq_model resolved to {diq_model.__file__}, outside {REPO_ROOT}"
)
print("diq_model", diq_model.__version__, "from", diq_model.__file__)

# Drop stages whose dependencies are unusable rather than failing later.
dropped = {
    stage: [d for d in deps if not probe(d)]
    for stage, deps in STAGE_DEPS.items()
}
dropped = {k: v for k, v in dropped.items() if v}
STAGES = [s for s in STAGES if s not in dropped]
if dropped:
    print("SKIPPING stages (dependency unusable):")
    for stage, deps in dropped.items():
        print(f"  {stage} needs {', '.join(deps)}")
print("stages to run:", ", ".join(STAGES))


## 2. Data — real Recruit download from Kaggle (idempotent)

The four CSVs are the model’s actual inputs:

| File | Role |
| --- | --- |
| `air_visit_data.csv` | demand target — store × visit date × visitors |
| `air_store_info.csv` | static store attributes (genre, area) |
| `date_info.csv` | Japanese holiday calendar |
| `air_reserve.csv` | reservation logs — partial by design (~1/3 of stores) |

The cell only downloads when the files are absent, so a recycled Colab runtime does not re-fetch. It reads `kaggle.json` from Drive at `MyDrive/kaggle/kaggle.json` — that is the file you add before running. If you prefer not to use the Kaggle API, place the four CSVs directly into `DATA_ROOT` and skip this cell’s download path.


In [ ]:
# ==============================================================================
# WHAT THIS CELL DOES:
#   - If all 4 Recruit CSVs already exist in DATA_ROOT, it does nothing.
#   - Otherwise it copies kaggle.json into ~/.kaggle, downloads the Kaggle zip,
#     unzips it into DATA_ROOT, then asserts the 4 CSVs are really there.
#
# KEY FUNCTION _ensure_kaggle_token():
#   Location order: ~/.kaggle/kaggle.json > Drive kaggle.json > fail loudly.
#   It copies + chmods the token so the kaggle CLI can use it non-interactively.
#
# IDEMPOTENCE:
#   Safe to re-run after a Colab recycle: downloads only when files are missing.
#
# ALTERNATIVE IF YOU SKIP KAGGLE:
#   Place the 4 CSVs directly into DATA_ROOT before running this cell, and the
#   download branch is skipped. The rest of the notebook does not care how they
#   got there.
# WHAT THIS CELL DOES:
#   - If all 4 Recruit CSVs already exist in DATA_ROOT, it does nothing.
#   - Otherwise it copies kaggle.json into ~/.kaggle, downloads the Kaggle zip,
#     unzips it into DATA_ROOT, then asserts the 4 CSVs are really there.
#
# KEY FUNCTION _ensure_kaggle_token():
#   Location order: ~/.kaggle/kaggle.json > Drive kaggle.json > fail loudly.
#   It copies + chmods the token so the kaggle CLI can use it non-interactively.
#
# IDEMPOTENCE:
#   Safe to re-run after a Colab recycle: downloads only when files are missing.
#
# ALTERNATIVE IF YOU SKIP KAGGLE:
#   Place the 4 CSVs directly into DATA_ROOT before running this cell, and the
#   download branch is skipped. The rest of the notebook does not care how they
#   got there.
# DATA — download the real Recruit CSVs from Kaggle only when missing.
# ==============================================================================
import shutil
import subprocess
from pathlib import Path

kaggle_json = Path("/content/drive/MyDrive/kaggle/kaggle.json") if IS_COLAB else None

def _ensure_kaggle_token() -> None:
    home_kaggle = Path.home() / ".kaggle" / "kaggle.json"
    if home_kaggle.exists():
        return
    if kaggle_json and kaggle_json.exists():
        home_kaggle.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(kaggle_json, home_kaggle)
        home_kaggle.chmod(0o600)
        return
    raise FileNotFoundError(
        "kaggle.json not found. Put it in Drive at MyDrive/kaggle/kaggle.json, "
        "or place the 4 CSVs directly into DATA_ROOT."
    )

if all((DATA_ROOT / f).exists() for f in RECRUIT_CSVS):
    print("Recruit already present:", DATA_ROOT)
else:
    _ensure_kaggle_token()
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    zip_dir = Path("/tmp/recruit")
    zip_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [sys.executable, "-m", "kaggle", "competitions", "download",
         "-c", KAGGLE_COMPETITION, "-p", str(zip_dir)],
        check=True,
    )
    zips = sorted(zip_dir.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(f"no zip downloaded into {zip_dir}")
    subprocess.run(["unzip", "-o", "-q", str(zips[0]), "-d", str(DATA_ROOT)], check=True)

missing = [f for f in RECRUIT_CSVS if not (DATA_ROOT / f).exists()]
assert not missing, f"missing Recruit files: {missing}"

print("Recruit files on disk:")
for f in RECRUIT_CSVS:
    print(f"  {(DATA_ROOT / f).stat().st_size / 1e6:7.2f} MB  {f}")


## 3. Audit the real data

Before anything trains, the pipeline runs the Section 3.7.1 audit: duplicates, parse failures, negative visitors, identifier consistency, temporal span, coverage. Hard checks block training; reference checks compare against the audited full-dataset figures.

This is the same audit the `ingest` stage runs, presented here explicitly so the data-quality story is visible before the metrics.


In [ ]:
# ==============================================================================
# WHAT THIS CELL DOES:
#   1. Loads the 4 CSVs through diq_model.data.load (with pinned dtypes).
#   2. Runs the Section 3.7.1 audit (duplicates, parse failures, negatives,
#      identifier consistency, temporal span, coverage).
#   3. Prints the human-readable audit report.
#   4. If STRICT=True and a hard check failed, it raises SystemExit before any
#      training code runs.
#
# KEY FUNCTION audit_recruit():
#   Returns an AuditReport with one entry per check, each as (passed, detail).
#   Hard checks: duplicate_store_days, date_parse_failures, visitor_value_range,
#   identifier_consistency. Reference checks: audited_scale, temporal_span.
#
# WHY THIS RUNS BEFORE TRAINING:
#   The pipeline's 'ingest' stage would run the same audit; this cell surfaces
#   the result early so a bad download fails cheaply and visibly.
# WHAT THIS CELL DOES:
#   1. Loads the 4 CSVs through diq_model.data.load (with pinned dtypes).
#   2. Runs the Section 3.7.1 audit (duplicates, parse failures, negatives,
#      identifier consistency, temporal span, coverage).
#   3. Prints the human-readable audit report.
#   4. If STRICT=True and a hard check failed, it raises SystemExit before any
#      training code runs.
#
# KEY FUNCTION audit_recruit():
#   Returns an AuditReport with one entry per check, each as (passed, detail).
#   Hard checks: duplicate_store_days, date_parse_failures, visitor_value_range,
#   identifier_consistency. Reference checks: audited_scale, temporal_span.
#
# WHY THIS RUNS BEFORE TRAINING:
#   The pipeline's 'ingest' stage would run the same audit; this cell surfaces
#   the result early so a bad download fails cheaply and visibly.
# AUDIT — read-only Section 3.7.1 checks on the real dataset.
# ==============================================================================
from diq_model.data.audit import audit_recruit
from diq_model.data.load import load_recruit

data = load_recruit(DATA_ROOT)
report = audit_recruit(data)
print(report.summary())

hard = {"duplicate_store_days", "date_parse_failures", "visitor_value_range", "identifier_consistency"}
if STRICT and not report.ok:
    blocking = set(report.failures) & hard
    if blocking:
        raise SystemExit(f"Audit blocked training: {', '.join(sorted(blocking))}")

audit_rows = {
    name: {"passed": passed, "detail": detail}
    for name, (passed, detail) in report.checks.items()
}
import pandas as pd
audit_df = pd.DataFrame(audit_rows).T
audit_df.index.name = "check"
audit_df


## 4. Data exploration — raw Recruit

This section is narrated on purpose: each figure answers one concrete question about the data the model will eventually see. Nothing here trains a model; it is all descriptive, and everything is computed from the ingested CSVs — not from hand-copied numbers.

The goal is to make the data tangible before it becomes features: how many stores, how many days, how skewed is demand, where does the weekly rhythm show up, and how partial is the reservation feed.


In [ ]:
# ==============================================================================
# EXPLORATION 1 — raw demand shape.
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

visits = data.visits.copy()
stores = data.stores.copy()
calendar = data.calendar.copy()
reservations = data.reservations.copy()

print("visits      :", f"{len(visits):,} rows")
print("stores      :", f"{stores['air_store_id'].nunique()} distinct")
print("date span   :", visits["visit_date"].min().date(), "->", visits["visit_date"].max().date())
print("reservations:", f"{len(reservations):,} rows" if len(reservations) else "none")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(visits["visitors"], bins=60)
axes[0].set_title("raw daily visitors (right-skewed)")
axes[0].set_xlabel("visitors/day")
axes[1].hist(np.log1p(visits["visitors"]), bins=60)
axes[1].set_title("log1p(visitors) — the training target")
axes[1].set_xlabel("log1p(visitors)")
by_dow = visits.groupby(visits["visit_date"].dt.dayofweek)["visitors"].mean()
axes[2].bar(range(7), by_dow.values)
axes[2].set_title("mean demand by weekday")
axes[2].set_xlabel("day of week (0 = Monday)")
axes[2].set_xticks(range(7))
fig.suptitle("Figure 1: target distribution and the weekly rhythm", y=1.02)
fig.tight_layout()
plt.show()

summary = {
    "rows": int(len(visits)),
    "stores": int(stores["air_store_id"].nunique()),
    "date_min": str(visits["visit_date"].min().date()),
    "date_max": str(visits["visit_date"].max().date()),
    "visitors_mean": round(float(visits["visitors"].mean()), 2),
    "visitors_median": round(float(visits["visitors"].median()), 2),
    "visitors_max": int(visits["visitors"].max()),
    "visitors_p95": round(float(visits["visitors"].quantile(0.95)), 2),
    " reservation_rows": int(len(reservations)),
    " reservation_stores": int(reservations["air_store_id"].nunique()) if len(reservations) else 0,
}
summary


### 4.1 Coverage and missingness

Demand is not observed every day for every store. This is real, not a bug to hide: the pipeline treats missing days as missing, never as zero, and the feature builder keeps the corresponding nulls rather than imputing.


In [ ]:
# ==============================================================================
# EXPLORATION 2 — per-store coverage and missing days.
# ==============================================================================
obs_per_store = visits.groupby("air_store_id")["visit_date"].nunique()
span_per_store = visits.groupby("air_store_id")["visit_date"].agg(["min", "max"])
span_days = (span_per_store["max"] - span_per_store["min"]).dt.days + 1
coverage = (obs_per_store / span_days).rename("coverage")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(coverage.clip(lower=0), bins=40)
axes[0].set_title("per-store observed-day coverage")
axes[0].set_xlabel("observed days / span days")
axes[1].hist(obs_per_store.clip(upper=400), bins=40)
axes[1].set_title("observed days per store")
axes[1].set_xlabel("observed days")
fig.tight_layout()
plt.show()

print("stores with < 80% coverage:", int((coverage < 0.80).sum()))
print("median observed days/store:", int(obs_per_store.median()))
print("min / max observed days:", int(obs_per_store.min()), int(obs_per_store.max()))

coverage_frame = pd.DataFrame({
    "store": coverage.index,
    "observed_days": obs_per_store.values,
    "span_days": span_days.values,
    "coverage": coverage.round(3).values,
}).sort_values("coverage")
coverage_frame.head(10)


### 4.2 Calendar signal

The calendar file is small but important: it drives the `is_holiday`, `day_before_holiday`, `day_after_holiday` features, including the long-weekend interactions added in feature set `fs-2026.09.2`.


In [ ]:
# ==============================================================================
# EXPLORATION 3 — holiday calendar.
# ==============================================================================
cal = calendar.copy()
print("calendar rows:", len(cal))
print("holidays     :", int(cal["holiday_flg"].sum()))

holiday_dates = cal.loc[cal["holiday_flg"] == 1, "calendar_date"]
print("first 10 holidays:")
print(holiday_dates.head(10).dt.date.to_string())

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(cal["calendar_date"], cal["holiday_flg"], drawstyle="steps-post")
ax.set_title("holiday calendar (1 = holiday)")
ax.set_ylabel("holiday_flg")
fig.tight_layout()
plt.show()


### 4.3 Reservation feed — partial by design

Reservation features are supplementary (Section 3.10). They are only available for part of the estate, and only for bookings made strictly before the visit day. The model must not depend on them, so the exploration here states the coverage honestly rather than making it look more complete than it is.


In [ ]:
# ==============================================================================
# EXPLORATION 4 — reservation coverage.
# ==============================================================================
if len(reservations):
    rsv_stores = set(reservations["air_store_id"].unique())
    visit_stores = set(visits["air_store_id"].unique())
    rsv_in_visits = rsv_stores & visit_stores
    print("reservation rows   :", len(reservations))
    print("reservation stores :", len(rsv_stores))
    print("reservation stores that also have visits:", len(rsv_in_visits))
    print("share of visit stores with a reservation feed:",
          round(100 * len(rsv_in_visits) / len(visit_stores), 1), "%")

    rsv = reservations.copy()
    rsv["visit_date"] = rsv["visit_datetime"].dt.normalize()
    pre_day = rsv.loc[rsv["reserve_datetime"] < rsv["visit_datetime"]]
    print("reservations booked strictly before the visit day:", len(pre_day))
    print("share pre-day:", round(100 * len(pre_day) / max(len(reservations), 1), 1), "%")
else:
    print("no reservation file present — reservation features will be absent")


## 5. Train the full pipeline

This runs the identical `run_pipeline(...)` the CLI and the existing Colab notebook use. The stages are:

1. `ingest` — load + audit + persist normalized frames; pin source SHA-256.
2. `baseline` — five chronological baselines; the best is the bar to beat.
3. `features` — leakage-safe 26-feature frame (`fs-2026.09.2`); gates fail loudly.
4. `fallbacks` — Linear / RF / XGBoost on the same frame and split.
5. `lightgbm` — primary model; chronological early stopping on the train tail.
6. `catboost` — second primary; same split, same rows, different failure mode.
7. `conformal` — stratified adaptive conformal intervals; 90% target.
8. `policy` — four preparation policies; multiplier/risk tuned on validation.
9. `shap` — per-forecast TreeSHAP; stored, not recomputed.
10. `sensitivity` — leave-one-group-out feature ablation + no-reservations variant.
11. `backtest` — 6-fold rolling-origin refits.

Every stage writes an atomic checkpoint before the next starts. With `FORCE=False`, a re-run skips finished stages and reloads them — that is what makes a Colab recycle painless.


In [ ]:
# ==============================================================================
# WHAT THIS CELL DOES:
#   1. Opens the CheckpointStore at CKPT_DIR.
#   2. Calls run_pipeline(...) with the exact stage list, seed, split and flags.
#   3. Each stage writes an atomic checkpoint before the next one starts.
#   4. With FORCE=False, already-finished stages are skipped and reloaded.
#   5. Returns a dict: results[stage_name] = metrics_dict.
#
# KEY FUNCTION run_pipeline():
#   Located in model/src/diq_model/train.py. Fixed stage order; completed
#   stages are loaded from manifest.json and skipped on resume.
#
# KEY CLASS CheckpointStore:
#   Located in model/src/diq_model/artifacts.py. Filesystem store with atomic
#   writes (temp file + rename) and a per-run manifest.json index.
#
# WHAT YOU GET BACK:
#   results is the dict every report cell below reads from, e.g.
#     results['lightgbm'] -> {'mae': ..., 'best_iteration': ..., ...}
#     results['conformal'] -> {'empirical_coverage': ..., ...}
#     results['sensitivity'] -> {'feature_group_ablation': ..., ...}
#     results['backtest'] -> {'n_folds': ..., 'mae_mean': ..., ...}
# WHAT THIS CELL DOES:
#   1. Opens the CheckpointStore at CKPT_DIR.
#   2. Calls run_pipeline(...) with the exact stage list, seed, split and flags.
#   3. Each stage writes an atomic checkpoint before the next one starts.
#   4. With FORCE=False, already-finished stages are skipped and reloaded.
#   5. Returns a dict: results[stage_name] = metrics_dict.
#
# KEY FUNCTION run_pipeline():
#   Located in model/src/diq_model/train.py. Fixed stage order; completed
#   stages are loaded from manifest.json and skipped on resume.
#
# KEY CLASS CheckpointStore:
#   Located in model/src/diq_model/artifacts.py. Filesystem store with atomic
#   writes (temp file + rename) and a per-run manifest.json index.
#
# WHAT YOU GET BACK:
#   results is the dict every report cell below reads from, e.g.
#     results['lightgbm'] -> {'mae': ..., 'best_iteration': ..., ...}
#     results['conformal'] -> {'empirical_coverage': ..., ...}
#     results['sensitivity'] -> {'feature_group_ablation': ..., ...}
#     results['backtest'] -> {'n_folds': ..., 'mae_mean': ..., ...}
# TRAIN — full checkpointed pipeline.
# ==============================================================================
import time
from diq_model.artifacts import CheckpointStore
from diq_model.train import run_pipeline

store = CheckpointStore(CKPT_DIR)
print("Completed stages so far:", store.completed_stages() or "none")

t0 = time.time()
results = run_pipeline(
    store=store,
    data_root=DATA_ROOT,
    test_start=TEST_START,
    seed=SEED,
    stages=STAGES,
    force=FORCE,
    strict=STRICT,
    n_folds=BACKTEST_FOLDS,
    test_days=BACKTEST_TEST_DAYS,
    advance_days=BACKTEST_ADVANCE_DAYS,
    min_train_days=BACKTEST_MIN_TRAIN_DAYS,
)
print(f"\npipeline finished in {time.time() - t0:.1f}s")
print("Completed stages:", ", ".join(sorted(results)) or "none")


## 6. Data exploration — feature frame

After training, this section inspects the leakage-safe feature frame the model actually trained on. The point is to show the information boundary concretely: what features exist, how they are distributed, where missingness comes from, and how the chronological split divides train from test. Nothing here recomputes metrics; it inspects the artifacts the run already wrote.


In [ ]:
# ==============================================================================
# EXPLORATION 5 — the trained feature frame.
# ==============================================================================
import joblib
import pandas as pd

features = store.load_dataframe("features.parquet")
meta = store.load_object("features_meta.joblib")
FEATURE_COLUMNS = meta["feature_columns"]
FEATURE_SET_VERSION = meta["feature_set_version"]
SPLIT = pd.Timestamp(TEST_START)

n_train = int((features["visit_date"] < SPLIT).sum())
n_test = int((features["visit_date"] >= SPLIT).sum())

print("feature set:", FEATURE_SET_VERSION)
print("rows:", f"{len(features):,}")
print("features:", len(FEATURE_COLUMNS))
print("stores:", features["air_store_id"].nunique())
print("date span:", features["visit_date"].min().date(), "->", features["visit_date"].max().date())
print("chronological split at", SPLIT.date(), ": train", f"{n_train:,}", "| test", f"{n_test:,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(features["visitors"], bins=60)
axes[0].set_title("target in the feature frame (visitors)")
axes[0].set_xlabel("visitors/day")
missing = features[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing):
    axes[1].barh(missing.index[::-1], missing.values[::-1])
    axes[1].set_title("missing values per feature (structural, never imputed)")
    axes[1].set_xlabel("rows with NaN")
fig.tight_layout()
plt.show()

pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "non_null_rows": features[FEATURE_COLUMNS].notna().sum().values,
    "mean": features[FEATURE_COLUMNS].mean().round(3).values,
}).T


### 6.1 Feature groups

The feature columns are declared once, in `diq_model.features.build.FEATURE_GROUPS`, and the model’s input list is derived from that declaration. That is what makes the sensitivity ablation and the production columns stay in sync.


In [ ]:
# ==============================================================================
# EXPLORATION 6 — feature groups and leakage discipline.
# ==============================================================================
from diq_model.features.build import FEATURE_GROUPS

frame = pd.DataFrame({
    "group": [g for g, cols in FEATURE_GROUPS.items() for _ in cols],
    "feature": [c for cols in FEATURE_GROUPS.values() for c in cols],
})
present = [c for c in frame["feature"] if c in features.columns]
frame = frame[frame["feature"].isin(present)]
frame["in_frame"] = True
frame


### 6.2 Train vs test demand profile

A quick look at whether the held-out window is comparable to the training window. The model is scored once on the test window and tuned nowhere on it, so this is descriptive, not evaluative.


In [ ]:
# ==============================================================================
# EXPLORATION 7 — train vs test demand profile.
# ==============================================================================
train = features[features["visit_date"] < SPLIT]
test = features[features["visit_date"] >= SPLIT]

def profile(df):
    return {
        "rows": int(len(df)),
        "stores": int(df["air_store_id"].nunique()),
        "mean_visitors": round(float(df["visitors"].mean()), 2),
        "median_visitors": round(float(df["visitors"].median()), 2),
        "max_visitors": int(df["visitors"].max()),
        "p95_visitors": round(float(df["visitors"].quantile(0.95)), 2),
    }

pd.DataFrame({"train": profile(train), "test": profile(test)}).T


## 7. Report — results chapter (mirrors DemandIQ-02-Results.ipynb)

Every number below is read from the run manifest and the stage artifacts, never from hand-copied values. That is the provenance rule the pipeline enforces: any figure traces to a stage, a seed, and a library set.

The structure mirrors the results notebook: per-model metrics, the baseline comparison, the interval quality, the policy trade-off, the SHAP top features, and the provenance.


In [ ]:
# ==============================================================================
# WHAT THIS CELL DOES:
#   Builds one metrics row per model from the results dict, then sorts by MAE.
#   The row comes from the stage metrics the pipeline recorded, not from memory.
#
# KEY FUNCTION _metrics():
#   A small helper that selects the common metric keys from a stage's metrics
#   dict. It uses .get() so a missing key becomes None rather than crashing.
#
# WHAT IS IN results[stage]['metrics'] for the main stages:
#   baseline  -> {baselines: {...}, best_baseline, best_baseline_mae, ...}
#   lightgbm  -> {mae, rmse, rmsle, mase, bias, r2, best_iteration, mae_lift_vs_baseline, ...}
#   catboost  -> {mae, rmse, rmsle, mase, bias, r2, mae_delta_vs_lightgbm, ...}
#   fallbacks -> {fallbacks: {linear_regression: {...}, random_forest: {...}, xgboost: {...}}, ...}
#
# HOW THE BEST BASELINE IS SHOWN:
#   results['baseline']['best_baseline'] names it; the cell pulls that baseline's
#   own row from results['baseline']['baselines'] and adds it to the table.
# WHAT THIS CELL DOES:
#   Builds one metrics row per model from the results dict, then sorts by MAE.
#   The row comes from the stage metrics the pipeline recorded, not from memory.
#
# KEY FUNCTION _metrics():
#   A small helper that selects the common metric keys from a stage's metrics
#   dict. It uses .get() so a missing key becomes None rather than crashing.
#
# WHAT IS IN results[stage]['metrics'] for the main stages:
#   baseline  -> {baselines: {...}, best_baseline, best_baseline_mae, ...}
#   lightgbm  -> {mae, rmse, rmsle, mase, bias, r2, best_iteration, mae_lift_vs_baseline, ...}
#   catboost  -> {mae, rmse, rmsle, mase, bias, r2, mae_delta_vs_lightgbm, ...}
#   fallbacks -> {fallbacks: {linear_regression: {...}, random_forest: {...}, xgboost: {...}}, ...}
#
# HOW THE BEST BASELINE IS SHOWN:
#   results['baseline']['best_baseline'] names it; the cell pulls that baseline's
#   own row from results['baseline']['baselines'] and adds it to the table.
# REPORT 1 — metrics table for every model in the run.
# ==============================================================================
def _metrics(name, m):
    return {"model": name, **{k: m.get(k) for k in ("mae", "rmse", "rmsle", "mase", "bias", "r2", "test_rows")}}

rows = []
for name, m in sorted(results.get("fallbacks", {}).get("fallbacks", {}).items()):
    rows.append(_metrics(f"fallback:{name}", m))
for name in ("lightgbm", "catboost"):
    if name in results:
        rows.append(_metrics(name, results[name]))
best_base = results.get("baseline", {}).get("best_baseline")
if best_base and "baselines" in results.get("baseline", {}):
    rows.append(_metrics(f"baseline:{best_base}", results["baseline"]["baselines"][best_base]))

perf = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
lgb = results.get("lightgbm", {})
lift = lgb.get("mae_lift_vs_baseline")

print(f"LightGBM MAE {lgb.get('mae')} | best baseline {best_base} {lgb.get('baseline_mae')} | lift {lift:.1%}")
print(f"MASE {lgb.get('mase')} (< 1 beats seasonal naive) | best_iteration {lgb.get('best_iteration')}")
perf


### 7.1 Baseline comparison

The baseline stage builds five deliberately simple models and takes the best seriously. A model that does not beat `best_baseline` has learned nothing beyond what a simple rule already gives.


In [ ]:
# ==============================================================================
# REPORT 2 — baseline ranking.
# ==============================================================================
if "baseline" in results and "baselines" in results["baseline"]:
    bl = results["baseline"]["baselines"]
    bl_df = pd.DataFrame(bl).T
    bl_df = bl_df[["mae", "rmse", "rmsle", "mase", "bias", "test_rows"]]
    bl_df["lift_vs_best"] = (1 - bl_df["mae"] / bl_df["mae"].min()).round(4)
    bl_df.sort_values("mae", inplace=True)
    bl_df


### 7.2 Intervals — coverage and width

The conformal stage reports empirical coverage against the 90% target, mean interval width, coverage by demand level, coverage by predicted bucket, the global-quantile honesty baseline, and sensitivity at 80% and 95%. Thin buckets fall back to the global band, and that share is reported too.


In [ ]:
# ==============================================================================
# REPORT 3 — conformal intervals.
# ==============================================================================
conf = results.get("conformal", {})
if conf:
    print("target coverage:", conf.get("target_coverage"))
    print("empirical coverage:", conf.get("empirical_coverage"))
    print("mean interval width:", conf.get("mean_interval_width"), "visitors")
    print("global fallback share:", conf.get("global_fallback_share"))
    print("\nby realised demand level:")
    print(pd.DataFrame([conf.get("coverage_by_demand_level", {})], index=[0]).T)
    print("\nby predicted bucket (calibration view):")
    print(pd.DataFrame([conf.get("coverage_by_predicted_bucket", {})], index=[0]).T)
    print("\nsensitivity at other levels:")
    print(pd.DataFrame(conf.get("coverage_level_sensitivity", {})).T)
    print("\nglobal quantile comparison (honesty baseline):")
    gq = conf.get("global_quantile_comparison", {})
    print("  coverage:", gq.get("empirical_coverage"), "| width:", gq.get("mean_interval_width"))
    print("  by demand level:", gq.get("coverage_by_demand_level"))


### 7.3 Policy trade-off

Four policies convert forecasts into preparation quantities, and the simulation reports both sides of the trade-off: surplus (over-preparation) and unmet demand. The multiplier and risk weight are chosen on the validation window, never on the test window.


In [ ]:
# ==============================================================================
# REPORT 4 — policy trade-off.
# ==============================================================================
pol = results.get("policy", {})
if pol:
    print("uncertainty_scale:", pol.get("uncertainty_scale"))
    print("risk_weight:", pol.get("risk_weight"))
    print("verdict:", pol.get("scale_selection", {}).get("verdict"))
    print("\npolicies:")
    pol_df = pd.DataFrame(pol.get("policies", {})).T
    if not pol_df.empty:
        pol_df["surplus_ps_per_day"] = (pol_df["surplus_ps"] / pol_df["days"]).round(2)
        pol_df["unmet_ud_per_day"] = (pol_df["unmet_ud"] / pol_df["days"]).round(2)
    pol_df
    head = pol.get("headline", {})
    if head:
        print("\nheadline vs point forecast:")
        print("  unmet demand reduction:", f"{head.get('ud_reduction_vs_point', 0):.1%}")
        print("  surplus increase:", f"+{head.get('ps_increase_vs_point', 0):,.0f} visitor-equivalents")


### 7.4 SHAP — per-forecast attribution

TreeSHAP contributions are computed for the LightGBM production candidate on the same rows it scored, converted to visitor units, and stored. The top features and the additivity reconstruction error are reported openly. The disclaimer is mandatory: this is predictive attribution, not causal effect.


In [ ]:
# ==============================================================================
# WHAT THIS CELL DOES:
#   Reads results['shap'] and prints the headline explanation figures.
#   The SHAP values were computed and stored during the 'shap' stage,
#   so this cell only displays them; it does not recompute contributions.
#
# KEY FIELD results['shap']['disclaimer']:
#   The mandatory predictive-attribution caveat. It is printed verbatim here
#   because every explanation surface must show it.
#
# KEY FIELD results['shap']['median_relative_reconstruction_error']:
#   Additivity check in visitor units: prediction ≈ base_value + Σ contributions.
#   The residual is the log-space linearisation error, reported openly.
#
# WHERE THE ACTUAL SHAP VALUES LIVE:
#   CKPT_DIR / shap_contributions.parquet (long format: store, date, feature, impact).
#   The stage also saved shap_meta.joblib with the disclaimer + feature-set version.
# WHAT THIS CELL DOES:
#   Reads results['shap'] and prints the headline explanation figures.
#   The SHAP values were computed and stored during the 'shap' stage,
#   so this cell only displays them; it does not recompute contributions.
#
# KEY FIELD results['shap']['disclaimer']:
#   The mandatory predictive-attribution caveat. It is printed verbatim here
#   because every explanation surface must show it.
#
# KEY FIELD results['shap']['median_relative_reconstruction_error']:
#   Additivity check in visitor units: prediction ≈ base_value + Σ contributions.
#   The residual is the log-space linearisation error, reported openly.
#
# WHERE THE ACTUAL SHAP VALUES LIVE:
#   CKPT_DIR / shap_contributions.parquet (long format: store, date, feature, impact).
#   The stage also saved shap_meta.joblib with the disclaimer + feature-set version.
# REPORT 5 — SHAP top features.
# ==============================================================================
if "shap" in results:
    shp = results["shap"]
    print("rows explained:", shp.get("rows_explained"))
    print("features per forecast:", shp.get("features_per_forecast"))
    print("base value (visitors):", shp.get("base_value_visitors"))
    print("median relative reconstruction error:", shp.get("median_relative_reconstruction_error"))
    print("disclaimer:")
    print(" ", shp.get("disclaimer"))
    print("\ntop features (mean |SHAP|):")
    pd.DataFrame(shp.get("top_features", []))
else:
    print("shap stage skipped — shap was not importable in this environment")


### 7.5 Sensitivity — feature groups and reservation availability

The sensitivity stage answers the Section 3.25 question: how much does the result depend on each feature group, and does the model degrade when reservations are absent? Each subset refits the same primary model on the same split, so the only variable is the information available. Positive `mae_delta_vs_full` means the group carried signal; negative means removing it helped — both are publishable outcomes, not errors to hide.


In [ ]:
# ==============================================================================
# WHAT THIS CELL DOES:
#   Reads results['sensitivity'] and prints the feature-group ablation and
#   the reservation-availability verdict.
#
# KEY FUNCTIONALITY IN THE TRAINED STAGE:
#   - Leave-one-group-out: the same LightGBM refit with one FEATURE_GROUPS group
#     removed. mae_delta_vs_full > 0 means that group carried signal.
#   - No-reservations variant: refit without the reservation columns, to test
#     the product requirement that the model not degrade when bookings are absent.
#
# KEY FIELD results['sensitivity']['feature_group_ablation']:
#   One entry per group: dropped_features, defined, mae, mae_delta_vs_full, ...
#
# KEY FIELD results['sensitivity']['reservation_availability']:
#   reservation_features_present, defined, with/without_reservations_mae,
#   mae_delta_without, verdict.
# WHAT THIS CELL DOES:
#   Reads results['sensitivity'] and prints the feature-group ablation and
#   the reservation-availability verdict.
#
# KEY FUNCTIONALITY IN THE TRAINED STAGE:
#   - Leave-one-group-out: the same LightGBM refit with one FEATURE_GROUPS group
#     removed. mae_delta_vs_full > 0 means that group carried signal.
#   - No-reservations variant: refit without the reservation columns, to test
#     the product requirement that the model not degrade when bookings are absent.
#
# KEY FIELD results['sensitivity']['feature_group_ablation']:
#   One entry per group: dropped_features, defined, mae, mae_delta_vs_full, ...
#
# KEY FIELD results['sensitivity']['reservation_availability']:
#   reservation_features_present, defined, with/without_reservations_mae,
#   mae_delta_without, verdict.
# REPORT 6 — sensitivity ablation.
# ==============================================================================
if "sensitivity" in results:
    sen = results["sensitivity"]
    print("features full:", sen.get("features_full"))
    print("most important group:", sen.get("most_important_group"))
    print("\nfeature-group ablation (mae_delta_vs_full > 0 = group carried signal):")
    ab = pd.DataFrame([
        {
            "subset": f"without_{g}",
            "features": sen["features_full"] - info.get("dropped_features", 0),
            "mae": info.get("mae"),
            "mae_delta_vs_full": info.get("mae_delta_vs_full"),
        }
        for g, info in sen.get("feature_group_ablation", {}).items()
        if info.get("defined")
    ])
    full_row = {
        "subset": "full",
        "features": sen.get("features_full"),
        "mae": sen.get("full", {}).get("mae"),
        "mae_delta_vs_full": 0.0,
    }
    ab = pd.concat([ab, pd.DataFrame([full_row])], ignore_index=True)
    ab.sort_values("mae", inplace=True)
    ab
    rv = sen.get("reservation_availability", {})
    print("\nreservation availability:")
    for k, v in rv.items():
        print(f"  {k}: {v}")
else:
    print("sensitivity stage skipped")


### 7.6 Backtest — rolling-origin folds

The backtest scores six independent 30-day windows, each refitted from scratch, and reports mean and standard deviation. That is a stronger claim than one held-out window: it shows the result is not a lucky stretch. Each fold also carries its own interval, so coverage is reported beside MAE.


In [ ]:
# ==============================================================================
# WHAT THIS CELL DOES:
#   Reads results['backtest'] and then loads backtest_folds.parquet to show
#   the per-fold table.
#
# KEY FIELDS IN results['backtest']:
#   n_folds, mae_mean, mae_std, rmsle_mean, rmsle_std, coverage_mean.
#   These come from the backtest_summary.joblib the stage wrote.
#
# KEY FILE CKPT_DIR / backtest_folds.parquet:
#   One row per fold with train/test windows and per-fold metrics. Each fold
#   refits LightGBM from scratch, so the mean is over independent refits.
#
# WHY THE MEAN/STD MATTERS:
#   One held-out window is a sample of size one. Six folds show the result is
#   not a lucky stretch. Coverage is reported beside MAE for the same reason.
# WHAT THIS CELL DOES:
#   Reads results['backtest'] and then loads backtest_folds.parquet to show
#   the per-fold table.
#
# KEY FIELDS IN results['backtest']:
#   n_folds, mae_mean, mae_std, rmsle_mean, rmsle_std, coverage_mean.
#   These come from the backtest_summary.joblib the stage wrote.
#
# KEY FILE CKPT_DIR / backtest_folds.parquet:
#   One row per fold with train/test windows and per-fold metrics. Each fold
#   refits LightGBM from scratch, so the mean is over independent refits.
#
# WHY THE MEAN/STD MATTERS:
#   One held-out window is a sample of size one. Six folds show the result is
#   not a lucky stretch. Coverage is reported beside MAE for the same reason.
# REPORT 7 — backtest folds.
# ==============================================================================
if "backtest" in results:
    bt = results["backtest"]
    print("n_folds:", bt.get("n_folds"))
    print("mae_mean:", bt.get("mae_mean"), "±", bt.get("mae_std"))
    print("rmsle_mean:", bt.get("rmsle_mean"), "±", bt.get("rmsle_std"))
    print("coverage_mean:", bt.get("coverage_mean"))
    folds_path = CKPT_DIR / "backtest_folds.parquet"
    if folds_path.exists():
        folds = pd.read_parquet(folds_path)
        folds_df = folds.copy()
        for c in ("mae", "rmse", "rmsle", "mase", "bias", "coverage", "mean_width"):
            if c in folds_df.columns:
                folds_df[c] = folds_df[c].round(3)
        folds_df
else:
    print("backtest stage skipped")


## 8. Provenance

The manifest records, per stage, timestamps, durations, seed, Python version, platform, library versions, and the SHA-256 of each source CSV. Any figure in this notebook can be traced back to the exact bytes and code that produced it.


In [ ]:
# ==============================================================================
# PROVENANCE — timings, versions, seed, source checksums.
# ==============================================================================
import json

manifest = json.loads((CKPT_DIR / "manifest.json").read_text())
stages = manifest["stages"]
print(f"{len(stages)} stages recorded: {', '.join(sorted(stages))}")

ingest = stages.get("ingest", {})
print("python  :", ingest.get("python_version"), "on", ingest.get("platform"))
print("seed    :", ingest.get("seed"))
print("versions:", ingest.get("versions"))
print("\nsource SHA-256 (provenance):")
for name, digest in ingest.get("metrics", {}).get("source_sha256", {}).items():
    print(f"  {name:<24} {digest[:32]}...")

print("\nper-stage durations:")
for name in sorted(stages, key=lambda s: stages[s]["started_at"]):
    print(f"  {name:<12} {stages[name]['duration_s']:>8.1f}s")


## 9. Export the serving bundle

The backend does not import the training package. It loads one file: `serving_bundle.joblib`, written by `model/scripts/export_serving_bundle.py` from the run directory. This notebook runs that exact export, so the artifact it produces is the artifact the API is ready to consume.

The bundle contains the native LightGBM booster, the exact feature-column order, the feature-set version, the final conformal state, the holiday calendar the feature flags assume, the run’s point-accuracy metrics, the comparison models, the backtest folds (when present), and the provenance.


In [ ]:
# ==============================================================================
# BACKEND SERVING PATH CONTRACT (backend/app/services/forecasting/bundle.py):
#
# The backend does NOT import the training package. It loads one file:
#   serving_bundle.joblib   (format='demandiq-serving-bundle', bundle_version=1)
#
# Resolution order the backend uses:
#   1. MODEL_BUNDLE_PATH env var  (if set and loadable)
#   2. default candidates, probed in order:
#        ../model/artifacts/runs/latest/serving_bundle.joblib
#        /model/artifacts/runs/latest/serving_bundle.joblib
#        model/artifacts/runs/latest/serving_bundle.joblib
#
# This notebook writes the bundle into CKPT_DIR, which is NOT one of those
# defaults on Colab. So to serve THIS run you either:
#   (a) set MODEL_BUNDLE_PATH to the file this notebook wrote, or
#   (b) copy/symlink it into one of the default locations above.
#
# On Colab that means, before starting/restarting the backend:
#   export MODEL_BUNDLE_PATH='/content/drive/MyDrive/demandiq/artifacts/runs/nb03/serving_bundle.joblib'
# or, if you want bundle discovery by default (local deploy style):
#   cp /content/drive/MyDrive/demandiq/artifacts/runs/nb03/serving_bundle.joblib ../model/artifacts/runs/latest/
#
# The cell below prints the exact MODEL_BUNDLE_PATH value you should set for this run.
# WHAT THIS CELL DOES:
#   1. Adds model/scripts to sys.path.
#   2. Calls export_bundle(CKPT_DIR) from export_serving_bundle.py.
#   3. Dumps the returned dict as serving_bundle.joblib.
#   4. Prints a human-readable summary of the bundle it just wrote.
#
# KEY FUNCTION export_bundle():
#   Located in model/scripts/export_serving_bundle.py. It reads ONLY the run
#   artifacts (manifest.json + stage parquets + joblib models), replays the
#   conformal loop to rebuild the serving state, and assembles the bundle.
#   It does not import the training package at serving time.
#
# WHAT THE BUNDLE CONTAINS:
#   model.booster, model.feature_columns, model.feature_set_version,
#   conformal state, calendar.holiday_dates, metrics, comparison models,
#   backtest (when present), provenance (seed, source SHA-256, rows, stores).
#
# WHY THIS IS THE BACKEND ARTIFACT:
#   The API does not import diq_model; it loads serving_bundle.joblib and reads
#   model.booster + model.feature_columns + conformal from it.
#   This cell writes exactly that file, from exactly this run.
# EXPORT — write serving_bundle.joblib from the run directory.
# ==============================================================================
import sys as _sys
from pathlib import Path

_sys.path.insert(0, str(REPO_ROOT / "model" / "scripts"))
from export_serving_bundle import export_bundle

bundle = export_bundle(CKPT_DIR)
out = CKPT_DIR / "serving_bundle.joblib"
import joblib
joblib.dump(bundle, out)

m = bundle["model"]
ev = bundle["metrics"]
c = bundle["conformal"]
print("serving bundle written:", out)
print(f"  model: {m['version']} ({len(m['feature_columns'])} features, fs {m['feature_set_version']})")
print(f"  metrics: MAE {ev['mae']} | RMSE {ev['rmse']} | RMSLE {ev['rmsle']} "
      f"(vs {ev['baseline_name']} MAE {ev['baseline_mae']}, lift {ev['mae_lift_vs_baseline']})")
print(f"  window: train {ev['train_start']}..{ev['train_end']} | "
      f"test {ev['test_start']}..{ev['test_end']} ({ev['test_rows']} store-days)")
print(f"  booster best_iteration: {m['best_iteration']}")
print("  comparison models:")
for name, entry in bundle.get("comparison", {}).items():
    cm = entry["metrics"]
    print(f"    {name} {entry['version']} MAE {cm['mae']} "
          f"(delta vs production {cm['mae_delta_vs_production']}), status {entry['status']}")
bt = bundle.get("backtest")
if isinstance(bt, dict):
    cov = bt.get("coverage_mean")
    cov_txt = f", mean coverage {cov:.3f}" if isinstance(cov, (int, float)) else ""
    print(f"  backtest: {bt['n_folds']} folds, mean MAE {bt['mae_mean']} "
          f"(±{bt['mae_std']}) vs {bt['baseline_name']} {bt['baseline_mae']}{cov_txt}")
else:
    print("  backtest: none in this run")
print(f"  conformal: static serving coverage {c['achieved_coverage']:.4f} @ width {c.get('achieved_width')}")
print(f"  strata alphas: {[s['alpha'] for s in c['strata']]} | global {c['global']['alpha']}")
print(f"  holiday_dates: {len(bundle['calendar']['holiday_dates'])}")


## 10. Gather downloadable artifacts

At the end of a Colab run, these are the artifacts you typically want to keep. The serving bundle is the one the backend consumes; the manifest and report are what let you explain and reproduce the run.


In [ ]:
# ==============================================================================
# BACKEND SERVING PATH (continued):
#
# The ONE file the backend consumes from this run is:
#   serving_bundle.joblib   (backend reads model.booster + feature_columns
#                             + conformal + metrics + comparison + backtest + provenance)
#
# To serve this run you must make that file reachable by the backend. Options:
#   (a) Set MODEL_BUNDLE_PATH to the bundle path printed by the EXPORT cell.
#   (b) Copy the bundle into a default-discoverable location:
#         ../model/artifacts/runs/latest/serving_bundle.joblib
#         /model/artifacts/runs/latest/serving_bundle.joblib   (docker compose mount)
#         model/artifacts/runs/latest/serving_bundle.joblib
#
# The cell below prints a ready-to-copy MODEL_BUNDLE_PATH for this run.
# WHAT THIS CELL DOES:
#   Lists the artifacts in CKPT_DIR and marks which ones exist, with a note on
#   what each one is for.
#
# CORE vs OPTIONAL:
#   Core = serving_bundle.joblib + manifest.json (the ones to keep/download).
#   Optional = the per-stage artifacts, useful for inspection or debugging.
#
# DOWNLOAD ON COLAB:
#   Either from the Drive file browser, or via files.download() for the bundle.
#   The cell prints the exact path so you can copy it from the output.
# ARTIFACTS — what to download before stopping the runtime.
# ==============================================================================
from pathlib import Path

artifacts = [
    ("serving_bundle.joblib", "backend artifact — the file the API loads"),
    ("manifest.json", "provenance — seed, versions, per-stage metrics, source SHA-256"),
]
optional = [
    ("lightgbm_model.joblib", "trained booster + provenance"),
    ("features.parquet", "leakage-safe feature frame"),
    ("shap_contributions.parquet", "per-forecast TreeSHAP (long format)"),
    ("lightgbm_predictions.parquet", "held-out point predictions"),
    ("conformal_predictions.parquet", "held-out intervals"),
    ("policy_results.joblib", "policy trade-off table"),
    ("backtest_folds.parquet", "rolling-origin folds (when backtest ran)"),
    ("sensitivity_ablation.parquet", "feature-group ablation (when sensitivity ran)"),
]

print("CORE artifacts in", CKPT_DIR, ":")
for name, why in artifacts:
    p = CKPT_DIR / name
    print(f"  {'YES' if p.exists() else 'NO '}  {name:<32} {why}")

print("\nOPTIONAL artifacts:")
for name, why in optional:
    p = CKPT_DIR / name
    print(f"  {'YES' if p.exists() else 'NO '}  {name:<32} {why}")

print("\nOn Colab, to download the serving bundle from the file browser:")
print("  Drive > demandiq/artifacts/runs/nb03/serving_bundle.joblib")
print("\nOr programmatically (Colab only):")
if IS_COLAB:
    from google.colab import files
    print("  from google.colab import files")
    print(f"  files.download(str({CKPT_DIR!r} / 'serving_bundle.joblib'))")
else:
    print("  (local run — artifacts are already on disk at", CKPT_DIR, ")")


## 11. Reading notes for the presenter

A few points worth stating plainly when you walk through this notebook:

- **This trains the same model the backend serves.** The notebook does not build a separate demo model; it drives `diq_model.train.run_pipeline` and then exports the bundle with `export_serving_bundle.py`. The artifact is the real one.
- **Data is real and downloaded from Kaggle.** The four Recruit CSVs are the actual inputs. The audit runs first and blocks on hard integrity failures, so a bad download fails before it trains.
- **Leakage is structural, not conventional.** Rolling windows use `shift(1)` so they never include the target day; there is a poison-probe validation gate; and the chronological split never shuffles. Section 3.26 is why.
- **The target is `log1p(visitors)`.** Demand is right-skewed (mean ≈ 21, max 877), so squared error on raw counts is dominated by a few huge days. `log1p` compresses that; predictions are inverted with `expm1` and clipped at zero.
- **Intervals are adaptive and stratified.** One global quantile collapses coverage on the busiest days; the Mondrian stratified ACI calibrates each predicted-demand level on its own scale. Coverage is reported empirically, not asserted.
- **Policy is explicit and transparent.** Preparation quantity is a business decision kept separate from the forecast, and the multiplier/risk weight are chosen on validation, never test.
- **SHAP is predictive attribution, not causation.** The disclaimer ships with every explanation and is shown verbatim.
- **Resume is built in.** Each stage writes an atomic checkpoint; re-running **Run all** skips finished stages and reloads them from Drive, so a runtime recycle loses nothing.
- **Backtest and sensitivity are opt-in here because they are expensive.** They refit the primary model many times; the notebook includes them because you asked for the full picture, but their cost is why the default pipeline does not run them every time.
